In [1]:
!pip install transformers torch scikit-learn pandas numpy tqdm --quiet

import re
import pandas as pd
import numpy as np
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from transformers import DistilBertTokenizer, DistilBertModel
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from tqdm import tqdm

# loading data

train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")


# cleaning the text

def clean_text(text):
    if pd.isna(text): return ""
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"[^a-zA-Z0-9\s]", " ", text)
    return text.lower().strip()

train['clean_text'] = train['catalog_content'].apply(clean_text)
test['clean_text'] = test['catalog_content'].apply(clean_text)

#extracting IPQ

def extract_ipq(text):
    m = re.search(r"(\d+)\s*(pack|pcs|piece|pieces|count|qty)", text)
    return int(m.group(1)) if m else 1

train['ipq'] = train['catalog_content'].apply(extract_ipq)
test['ipq'] = test['catalog_content'].apply(extract_ipq)

scaler = StandardScaler()
train_ipq = scaler.fit_transform(train[['ipq']])
test_ipq = scaler.transform(test[['ipq']])

# Log-transform target
train['price_log'] = np.log1p(train['price'])


#DISTILBERT TOKENIZER

tokenizer = DistilBertTokenizer.from_pretrained("distilbert-base-uncased")

#DATASET

class PriceDataset(Dataset):
    def __init__(self, texts, ipq, targets=None, max_len=256):
        self.texts = texts
        self.ipq = ipq
        self.targets = targets
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = tokenizer(self.texts[idx], max_length=self.max_len,
                        padding='max_length', truncation=True,
                        return_tensors='pt')
        item = {k:v.squeeze(0) for k,v in enc.items()}
        item['ipq'] = torch.tensor(self.ipq[idx], dtype=torch.float)
        if self.targets is not None:
            item['labels'] = torch.tensor(self.targets[idx], dtype=torch.float)
        return item


# bert model

class TextRegressor(nn.Module):
    def __init__(self, dropout=0.3):
        super().__init__()
        self.bert = DistilBertModel.from_pretrained("distilbert-base-uncased")
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(self.bert.config.hidden_size + 1, 1)

    def forward(self, input_ids, attention_mask, ipq):
        text_feat = self.bert(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state[:,0,:]
        if ipq.dim()==1: ipq = ipq.unsqueeze(1)
        x = torch.cat([text_feat, ipq], dim=1)
        x = self.dropout(x)
        return self.fc(x).squeeze(1)

#train/valid split

X_train, X_val, ipq_train_split, ipq_val_split, y_train, y_val = train_test_split(
    train['clean_text'].values,
    train_ipq,
    train['price_log'].values.astype(np.float32),
    test_size=0.1,
    random_state=42
)

train_dataset = PriceDataset(X_train, ipq_train_split, y_train)
val_dataset = PriceDataset(X_val, ipq_val_split, y_val)
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=8)


# training loop

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = TextRegressor().to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)
loss_fn = nn.MSELoss()
epochs = 3

def smape(y_true, y_pred):
    return 100*np.mean(2*np.abs(y_pred - y_true) / (np.abs(y_true) + np.abs(y_pred) + 1e-8))

for epoch in range(epochs):
    model.train()
    for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}"):
        optimizer.zero_grad()
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        ipq = batch['ipq'].to(device)
        labels = batch['labels'].to(device)
        outputs = model(input_ids, attention_mask, ipq)
        loss = loss_fn(outputs, labels)
        loss.backward()
        optimizer.step()

    # Validation
    model.eval()
    val_preds, val_targets = [], []
    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            ipq = batch['ipq'].to(device)
            labels = batch['labels'].to(device)
            outputs = model(input_ids, attention_mask, ipq)
            val_preds.extend(np.expm1(outputs.cpu().numpy()))
            val_targets.extend(np.expm1(labels.cpu().numpy()))
    print(f"Epoch {epoch+1} Validation SMAPE: {smape(np.array(val_targets), np.array(val_preds)):.4f}")

# predicting test set

test_dataset = PriceDataset(test['clean_text'].values, test_ipq)
test_loader = DataLoader(test_dataset, batch_size=8)

model.eval()
test_preds = []
with torch.no_grad():
    for batch in test_loader:
        outputs = model(batch['input_ids'].to(device),
                        batch['attention_mask'].to(device),
                        batch['ipq'].to(device))
        test_preds.extend(np.expm1(outputs.cpu().numpy()))

#saving

submission = pd.DataFrame({'sample_id': test['sample_id'],
                           'price': np.clip(test_preds, 0.01, None)})
submission.to_csv("test_out.csv", index=False)
print("File saved as test_out.csv")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

...

Epoch 1 Validation SMAPE: 52.7421


Epoch 2: 100%|██████████| 8438/8438 [25:37<00:00,  5.49it/s]


Epoch 2 Validation SMAPE: 49.1780


Epoch 3: 100%|██████████| 8438/8438 [25:37<00:00,  5.49it/s]


Epoch 3 Validation SMAPE: 47.7130
✅ File saved as test_out.csv
